# Tutorial 1: Learn about `requests`

### Kailyn Lau

Follow along with [this tutorial on RealPython](https://realpython.com/python-requests/). 
Complete the first 5 sections thoroughly (up to but not including User Other HTTP methods). Then jump to section Improve Performance to learn about some advanced tricks that you might need at some point when scraping websites that have a lot of content.

Use markdown headings as appropriate to enumerate sections.

### Get Started with Python's `Requests` Library

In [7]:
import requests
from requests.exceptions import HTTPError

### Make a GET request

In [8]:
response = requests.get("https://api.github.com") ## retrieve data from a specific source (in this case, Github's API)

### Inspect the Response

Status Codes:
 - 1xx: Indicates that the request was received and understood.
 - 2xx: Indicates that the action requested by the client was received, understood, and accepted.
 - 3xx: Indicates that the client must take additional action to complete the request.
 - 4xx: Intended for situations where the error seems to have been caused by the client.
 - 5xx: Occurs when the server fails to fulfill a request.

(generally, 200s and 300s are errors, but there are workarounds)

Payload: valuable info found in request body


In [9]:
# if response.status_code == 200:
#     print("Success!")
# elif response.status_code == 404:
#     print("Not Found.")
    
if response:
    print("Success!")
else:
    raise Exception(f"Non-success status code: {response.status_code}")

Success!


In [10]:
URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except Exception as err:
        print(f"Other error occurred: {err}")
    else:
        print("Success!")

Success!
HTTP error occurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


In [15]:
response = requests.get("https://api.github.com")
print(response.content)
print(response.text)
print(response.json)
print(response.headers)
print(response.headers["date"])

b'{"current_user_url":"https://api.github.com/user","current_user_authorizations_html_url":"https://github.com/settings/connections/applications{/client_id}","authorizations_url":"https://api.github.com/authorizations","code_search_url":"https://api.github.com/search/code?q={query}{&page,per_page,sort,order}","commit_search_url":"https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}","emails_url":"https://api.github.com/user/emails","emojis_url":"https://api.github.com/emojis","events_url":"https://api.github.com/events","feeds_url":"https://api.github.com/feeds","followers_url":"https://api.github.com/user/followers","following_url":"https://api.github.com/user/following{/target}","gists_url":"https://api.github.com/gists{/gist_id}","hub_url":"https://api.github.com/hub","issue_search_url":"https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}","issues_url":"https://api.github.com/issues","keys_url":"https://api.github.com/user/keys","label_sea

### Add Query String Parameters

(the values after the ? on a url)

In [16]:
import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": "language:python", "sort": "stars", "order": "desc"},
)

json_response = response.json()
popular_repositories = json_response["items"]
for repo in popular_repositories[:3]:
    print(f"Name: {repo['name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']}\n")
    
    

Name: public-apis
Description: A collective list of free APIs
Stars: 478004

Name: free-programming-books
Description: :books: Freely available programming books
Stars: 396346

Name: system-design-primer
Description: Learn how to design large-scale systems. Prep for the system design interview.  Includes Anki flashcards.
Stars: 368975



### Customize Request Headers

(hidden inside the HTTP request)

In [17]:
# example to highlight matching search terms in the results by specifying the text-match media type in the Accept header

import requests

response = requests.get(
    "https://api.github.com/search/repositories",
    params={"q": '"real python"'},
    headers={"Accept": "application/vnd.github.text-match+json"},
)

json_response = response.json()
first_repository = json_response["items"][0]
print(first_repository["text_matches"][0]["matches"])

[{'text': 'Real Python', 'indices': [23, 34]}]


### Improve Performance

In [19]:
# Must add a timeout - by default requests wait indefinitely
requests.get("https://api.github.com", timeout=1)

# a connection request that will fail because the wait time isn't long enough
# requests.get("https://api.github.com", timeout=0.01)

# can also add both a connect timeout (client establish connection) and a read timeout (after client has established a connection):
# requests.get("https://api.github.com", timeout=(3.05, 5))
# and will return either a ConnectTimeout or ReadTimeout exception, which are subclasses of the general Timeout

<Response [200]>

Session objects are a way to keep a connection open per session, i.e. by using the same authentication across multiple requests. You can do something like this:

```
import requests
from custom_token_auth import TokenAuth

TOKEN = "<YOUR_GITHUB_PA_TOKEN>"

with requests.Session() as session:
    session.auth = TokenAuth(TOKEN)

    first_response = session.get("https://api.github.com/user")
    second_response = session.get("https://api.github.com/user")

print(first_response.headers)
print(second_response.json())
```

A transport adapter allows you to retry requests, as requests won't do this automatically. You can do something like the following:

```
import requests
from requests.adapters import HTTPAdapter
from requests.exceptions import RetryError
from urllib3.util.retry import Retry

retry_strategy = Retry(
    total=2,
    status_forcelist=[429, 500, 502, 503, 504]
)
github_adapter = HTTPAdapter(max_retries=retry_strategy)

with requests.Session() as session:
    session.mount("https://api.github.com", github_adapter)
    try:
        response = session.get("https://api.github.com/")
    except RetryError as err:
        print(f"Error: {err}")
```